# ASD vs TD classification with Joint Fusion++ (GATv2 + FT-Transformer) and the effect of graph sparsification (Medium exercise)

This notebook is the second part **exercise** of the Joint Fusion++ (JF++) model. Select the `torch23` environment. It repeats the analysis of `Joint_Fusion.ipynb` (structural-only model, functional-only model, joint model, 10-fold cross-validation, hyperparameter search with Optuna) but replaces the two MLP encoders with graph / attention encoders:

```
structural (221)  ── FT-Transformer (feature tokens + [CLS]) ── embedding ──┐
                                                                            ├─ concat ── fusion head ── logit
functional (5253) ── per-subject brain graph (103 regions) ── GATv2 ─────────┘
```

The 5253 functional features are the upper triangle of the $103 \times 103$ functional-connectivity matrix, so each subject is a **graph**: 103 nodes (brain regions), one candidate edge per pair of regions. The graph can be **complete** (all 5253 connections) or **sparsified** with a per-subject **kNN rule**. In this notebook we will use:
1. **kNN strict** (with `k = 1`): only the connections kept by the kNN rule are edges, *and* the removed connections are set to 0 in the node features;
2. **complete**: the reference without sparsification.

**Protocol**

| Item | Choice |
|---|---|
| Outer CV | `StratifiedKFold(10, shuffle=True, random_state=13)` |
| Scaling | `RobustScaler` fitted **only on the training subjects** of each fold |
| Optimizer | **SGD + momentum** (+ L1 penalty), `lr`, `momentum`, `l1_lambda`, `batch_size` searched |
| Loss | weighted BCE (`pos_weight = n_TD / n_ASD`), `ReduceLROnPlateau`, early stopping on subjects drawn from the training fold |
| Hyperparameters | Optuna (TPE + median pruner, 3-fold inner CV), **repeated inside every outer fold** (nested CV) |

**Exercise**

This is the **MEDIUM** version of the exercise. It **includes all the Easy tasks**. Every task is a block

```python
# ---- EXERCISE ---- <level>
# what to do (+ hints)
...
# ------------------
```

Replace every `...` with your code. The notebook does not run until all the blocks are completed.

| Level | Section | Task |
|---|---|---|
| Easy | 0 | Number of functional features `N_FUNC` |
| Easy | 1 | Merge the structural and functional tables (`join`) and convert the features to numpy |
| Easy | 2 | `fit_scaler` (RobustScaler on the training subjects only) and the outer folds `OUTER` |
| Easy | 4 | Complete the missing parts of `FunctionalGAT` (norm, activation, readout) and of the `MorphologicalTransformer` (tokenizer, [CLS], embedding) |
| Easy | 5 | `make_optimizer` (SGD + momentum) and `predict_proba` (sigmoid of the logits) |
| Medium | 2 | Leakage check between training, early-stopping and test subjects of fold 0 |
| Medium | 3 | Write the whole `knn_keep` function (kNN sparsification rule) |
| Medium | 4 | Write the two `GATv2Conv` layers of `FunctionalGAT` using the PyG documentation |
| Medium | 4 | Write the `nn.MultiheadAttention` layer of `AttnBlock` |
| Medium | 4 | Define the encoders of every branch in `JointFusionModel`, `MorphOnlyModel` and `FuncOnlyModel` |
| Medium | 5 | Write `predict_proba` without hints |

**Folders.** This notebook lives in `exercise/` and reads the inputs from the folders above it: the data from `/envs/data/jfpp`, the best hyperparameters from `../params/<run>` and the pretrained weights from `../weights/<run>`. Everything it writes (new checkpoints, metrics, figures) goes to `./results` and `./figures` inside `exercise/`, so the provided files are never overwritten.

#### Before starting

> **Training and pretrained weights.** By default (`TRAIN = True`) Sections 7-9 **train** the structural, functional and joint models of the selected graph on every outer fold, with the best hyperparameters stored in `../params/<run>` (a few minutes on a GPU for the kNN-strict graph). The new checkpoints are written to `./results/weights/<run>` (inside `exercise/`), so the provided weights are never overwritten. Set `TRAIN = False` to evaluate the pretrained checkpoints in `../weights/<run>` instead.
>
> The Optuna search is long (10 folds × 100 trials × 3 inner trainings) and is **off** by default (`RUN_SEARCH = False`): the stored best parameters in `../params/<run>` are used.
>
> The comparison **kNN strict vs complete** (Section 10.4) always uses the provided params and weights in `../params/<run>` and `../weights/<run>` of both graphs.
>
> **Figures.** All the figures are saved as vector PDF and 600-dpi PNG in `./figures` (inside `exercise/`).

## 0. Setup

Imports and device. JF++ needs `torch_geometric` (for `GATv2Conv` and `global_mean_pool`). `optuna` and `shap` are imported lazily, only inside the (commented / optional) cells that use them.

In [ ]:
import os, copy, json, math, time, logging, warnings
from dataclasses import dataclass
logging.disable(logging.WARNING)
warnings.simplefilter(action='ignore', category=Warning)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy import stats
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, roc_curve
from torch_geometric.nn import GATv2Conv, global_mean_pool # doc here: https://pytorch-geometric.readthedocs.io/en/latest/

# print torch version, cuda and device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', DEVICE)

### Protocol constants and paths

* `GRAPH` selects the graph used in Sections 7-9: `'knn_strict'` (default, `k = 1`) or `'complete'`. Section 10 always loads **both**.
* `RUN_NAMES` are the run folders: `WEIGHTS_DIR/<run>` holds the checkpoints and `PARAMS_DIR/<run>` the best Optuna parameters of every fold. **Edit these names if your runs are called differently.**
* `TRAIN` (default `True`): Sections 7-9 train the models; `False` loads the pretrained checkpoints of `WEIGHTS_DIR/<run>`.
* `RUN_SEARCH` (default `False`): repeat the nested Optuna search instead of using the best parameters stored in `PARAMS_DIR/<run>`.
* `TRAIN_DIR` is where the training and search cells write new checkpoints (`TRAIN_DIR/weights/<run>`) and parameters (`TRAIN_DIR/params/<run>`), so the copied weights are never overwritten.
* `FIG_DIR` is where every figure is saved (PDF + PNG).
* **Exercise folder**: the inputs are read from the parent folder (`DATA_IN = /envs/data/jfpp`, `PARAMS_DIR = ../params`, `WEIGHTS_DIR = ../weights`); the outputs (`TRAIN_DIR`, `OUT_DIR`, `FIG_DIR`) stay inside `exercise/`.

In [ ]:
# Data and dimensions
DATA_IN = '/envs/data/jfpp' # Data directory (one level above exercise/), open it to check the data
# ---- EXERCISE ---- Easy
# Input features of the structural branch (221), of the functional branch (N_FUNC) and number of regions (103).
# N_FUNC is the number of connections between pairs of different regions, i.e. the entries of the upper triangle of
# the N_REGIONS x N_REGIONS connectivity matrix (diagonal excluded). Write it as a function of N_REGIONS.
N_STRUCT, N_REGIONS = 221, 103
N_FUNC = ...
# ------------------

# Cross-validation protocol
OUTER_SEED, N_OUTER = 13, 10 # OUTER_SEED is the seed for 10-fold StratifiedCrossValidation, N_OUTER is the number of folds (here, 10)
INNER_SEED, N_INNER = 42, 3 # INNER_SEED is the seed for hyperparameters optimization using for the training set only (nested), N_INNER is the number of folds
ES_FRACTION = 0.15 # early-stopping subjects drawn (stratified) from the training subjects
PATIENCE, MAX_EPOCHS = 15, 150 # Patience and Max Numbers of Epoch for training
LR_FACTOR, LR_PATIENCE = 0.1, 5 # ReduceLROnPlateau inputs

# Optuna
N_TRIALS, TRIAL_EPOCH_CAP, TRIAL_PATIENCE = 100, 50, 6
SAMPLER_SEED, N_STARTUP = 1, 3 # TPESampler(seed=1), MedianPruner(n_startup_trials=3)
BATCH_GRID = [16, 32, 64]
L1_RANGE = (1e-5, 1e-1)
FIXED_ARCH = dict(gat_heads=8, morph_heads=2, morph_depth=1) # Fixed parameters: GAT heads, Attn heads and morph depth

# Sparsification graph rule: k = 1, only the strongest connection for each node is retained
KNN_K = 1
GRAPH_CFG = {'knn_strict': dict(knn_k=KNN_K, strict=True),    # Sparse graph, removed connections set to 0.0 in the node feature
             'complete':   dict(knn_k=None, strict=False)}    # All 5253 connections, all features
GRAPH = 'knn_strict'
G = GRAPH_CFG[GRAPH]

# Where the pretrained weights and parameters are
WEIGHTS_DIR = '../../weights' # one level above exercise/
PARAMS_DIR = '../../params'   # one level above exercise/

RUN_NAMES = {'knn_strict': 'jfpp_knn_strict_sgd_l1', 'complete': 'jfpp_complete_sgd_l1'}
RUN_DIRS = {g: os.path.join(WEIGHTS_DIR, r)
            for g, r in RUN_NAMES.items()}
PARAM_DIRS = {g: os.path.join(PARAMS_DIR, r)
              for g, r in RUN_NAMES.items()}

# Train (default) or load the pretrained checkpoints in Sections 7-9. Section 10.4 always uses the pretrained ones.
TRAIN = True
RUN_SEARCH = False # True repeats the nested Optuna search (long), False uses the best params in PARAMS_DIR

TRAIN_DIR = './results' # Output of the training cells (inside exercise/)
TRAIN_WEIGHTS = os.path.join(TRAIN_DIR, 'weights', RUN_NAMES[GRAPH])
TRAIN_PARAMS = os.path.join(TRAIN_DIR, 'params', RUN_NAMES[GRAPH])
OUT_DIR = './results' # Metrics written by this notebook
FIG_DIR = './figures' # Figures (PDF + PNG) written by this notebook

# For plotting AUC
INTERP_FPR = np.linspace(0, 1, 100)
EVAL_BS = 16 # Inference batch (a complete graph of 16 subjects has 168k directed edges)

np.random.seed(1); torch.manual_seed(1); torch.cuda.manual_seed_all(1)

### 0.1 Figure style

One style for every figure, sized for journals (single column 89 mm, double column 183 mm) with 7-pt sans-serif text (Arial, or the metric-compatible Liberation Sans). Colours are fixed per entity across the notebook and are colour-blind safe: **structural** aqua, **functional** orange, **joint** blue; for the graphs, **kNN strict** blue and **complete** grey. Connectivity values use a diverging blue-grey-red scale (negative / 0 / positive). `save_fig` writes every figure to `FIG_DIR` as vector PDF (editable text) and 600-dpi PNG.

In [ ]:
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

MM = 1 / 25.4                                   # matplotlib sizes are in inches
FIG_W1, FIG_W2 = 89 * MM, 183 * MM              # single / double column width

# Ink and chrome
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
# One colour per entity, the same in every figure
C_MODEL = {'struct': '#1baf7a', 'funct': '#eb6834', 'joint': '#2a78d6'}
C_GRAPH = {'complete': MUTED, 'knn_strict': '#2a78d6'}
LABEL_MODEL = {'struct': 'Structural (FT-Transformer)', 'funct': 'Functional (GATv2)', 'joint': 'Joint Fusion++'}
LABEL_MODEL_SHORT = {'struct': 'Structural', 'funct': 'Functional', 'joint': 'JF++'}
LABEL_GRAPH = {'complete': 'Complete', 'knn_strict': f'kNN strict (k = {KNN_K})'}
# Diverging scale for connectivity: blue (negative) - neutral grey (0) - red (positive)
CMAP_DIV = LinearSegmentedColormap.from_list('jfpp_div', ['#104281', '#3987e5', '#f0efec', '#e34948', '#8c1d1d'])

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'Nimbus Sans', 'DejaVu Sans'],
    'mathtext.fontset': 'dejavusans',
    'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
    'legend.fontsize': 6.5, 'axes.titlelocation': 'left', 'axes.titlepad': 5,
    'text.color': INK, 'axes.labelcolor': INK, 'axes.titlecolor': INK, 'axes.edgecolor': INK2,
    'axes.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.prop_cycle': mpl.cycler(color=[C_MODEL['joint'], C_MODEL['funct'], C_MODEL['struct']]),
    'xtick.color': INK2, 'ytick.color': INK2, 'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'xtick.minor.width': 0.4, 'ytick.minor.width': 0.4, 'xtick.minor.size': 1.5, 'ytick.minor.size': 1.5,
    'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.linestyle': '-',
    'lines.linewidth': 1.2, 'lines.markersize': 3.5, 'lines.solid_capstyle': 'round',
    'legend.frameon': False, 'legend.handlelength': 1.6, 'legend.borderaxespad': 0.3, 'legend.labelspacing': 0.35,
    'figure.dpi': 150, 'figure.facecolor': 'white', 'figure.constrained_layout.h_pad': 0.04,
    'figure.constrained_layout.w_pad': 0.04,
    'savefig.dpi': 600, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02, 'savefig.facecolor': 'white',
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
})

def ygrid(ax):
    """Recessive horizontal hairline grid behind the data."""
    ax.yaxis.grid(True, which='major'); ax.set_axisbelow(True)

def panel_label(ax, s, x=-0.2, y=1.02):
    """Bold lowercase panel letter at the top-left corner of the axes."""
    ax.text(x, y, s, transform=ax.transAxes, fontsize=9, fontweight='bold', va='bottom', ha='left', color=INK)

def save_fig(fig, name):
    """Save `fig` as FIG_DIR/name.png (600 dpi), then show it."""
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(os.path.join(FIG_DIR, f'{name}.png'))
    plt.show()

## 1. Data loading

Same tables and the same merge as `Joint_Fusion.ipynb` (harmonised ABIDE I + II features, inner join on `FILE_ID`, `DX_GROUP == -1` recoded to 0). The difference is that **nothing is scaled here**: `X` stays in its original units and the scaler is fitted inside every fold (Section 2). The unscaled functional values are also what the kNN rule ranks (Section 3).

Column layout of `X`: columns `0..220` structural, `221..5473` functional. Functional feature $k$ is the connection between regions `IU[k]` and `JU[k]`, in the column-major order of the upper triangle: $(0,1), (0,2), (1,2), (0,3), (1,3), (2,3), \dots$

In [ ]:
def load_data(data_in=DATA_IN):
    """Raw (harmonised, UNSCALED) features. DX_GROUP -1 -> 0 (TD = 0, ASD = 1)."""
    s = pd.read_csv(os.path.join(data_in, 'Harmonized_structural_features.csv')).set_index('FILE_ID') # structural features
    f = pd.read_csv(os.path.join(data_in, 'Harmonized_functional_features.csv')).set_index('FILE_ID') # functional features
    f = f.drop(['SITE', 'Database_Abide', 'AGE_AT_SCAN', 'DX_GROUP'], axis=1) # Remove not-feature cols
    # ---- EXERCISE ---- Easy
    # Merge the two tables on their index (FILE_ID), keeping only the subjects present in both tables, then move
    # FILE_ID back to a column. Hint: DataFrame.join(other, how=...) and DataFrame.reset_index()
    m = ...
    # ------------------
    m.loc[m.DX_GROUP == -1, 'DX_GROUP'] = 0 # Change DX_GROUP for TD from -1 to 0
    feats = m.drop(['SITE', 'Database_Abide', 'AGE_AT_SCAN', 'FILE_ID', 'DX_GROUP'], axis=1)
    # ---- EXERCISE ---- Easy
    # Convert the feature table `feats` to a numpy array of type np.float64
    X = ...
    # ------------------
    assert X.shape[1] == N_STRUCT + N_FUNC, X.shape
    y = m.DX_GROUP.to_numpy().astype(np.float32)
    assert set(np.unique(y)) == {0.0, 1.0}

    return {'X': X,
            'y': y,
            'site': m.SITE.to_numpy(),
            'file_id': m.FILE_ID.to_numpy(),
            'struct_names': list(feats.columns[:N_STRUCT]),
            'funct_names': list(feats.columns[N_STRUCT:])}

def feature_pairs(R=N_REGIONS):
    """Region pair (iu[k], ju[k]) of functional feature k, column-major: (0,1), (0,2), (1,2), (0,3), ..."""
    rows, cols = np.tril_indices(R, k=-1) # As the feature are in tabular form, we should find the ids
    return cols, rows

In [ ]:
D = load_data()
IU, JU = feature_pairs()
y_all = D['y']
print('subjects:', len(y_all), '| ASD:', int(y_all.sum()), '| TD:', int((1 - y_all).sum()), '| sites:', len(set(D['site'])))
print('X:', D['X'].shape, '| structural', N_STRUCT, '| functional', N_FUNC, '= pairs of', N_REGIONS, 'regions:',
      N_REGIONS * (N_REGIONS - 1) // 2)

### 1.1 Check of the feature order and region names (optional)

If `data/functional_features.csv` is available, we verify that the assumed pair order is consistent with its region codes (0 contradictions expected) and read the 103 region names for the plots. Without the file the regions are simply called `R0 ... R102`.

In [ ]:
def read_region_names(data_in=DATA_IN):
    """Region labels in node order and the number of ROI-code contradictions of the assumed pair order."""
    ff = pd.read_csv(os.path.join(data_in, 'functional_features.csv')) # here, you can find a CSV that map from IDs to ROI names
    assert len(ff) == N_FUNC
    code_of, bad = {}, 0
    name = [None] * N_REGIONS
    for a, b, c1, c2, n1, n2 in zip(IU, JU, ff['ROI 1'], ff['ROI 2'], ff['r1'], ff['r2']):
        bad += int(code_of.setdefault(a, c1) != c1) + int(code_of.setdefault(b, c2) != c2)
        name[a], name[b] = n1, n2
    return name, bad # This function returns ROI that are in all subjctes as after preprocessing they can be zero for several reasons

try:
    REGIONS, n_bad = read_region_names()
    print('Contradictions of the column-major pair order:', n_bad, '(must be 0)')
except Exception as e:
    REGIONS = [f'R{i}' for i in range(N_REGIONS)]
    print('region names not available, using R0..R102:', type(e).__name__)

## 2. Splits and scaling

- **Outer folds**: 10 stratified folds; each subject is a test subject exactly once. The split depends only on the labels and the seed, so the **same folds** are used by the three models and by the pretrained checkpoints.
- **Early-stopping subjects**: 15 % of the training subjects, used only to drive `ReduceLROnPlateau` and early stopping. The test fold is never used for training decisions (unlike the simplified protocol of the original JF notebook).
- **Inner folds**: a 3-fold split of the outer-training subjects, used by the Optuna objective.
- **RobustScaler**: fitted **only on the training subjects** of the fold (outer-train for the final model, inner-train inside the search) and then applied to the test subjects. Median and IQR of each feature therefore never include a test subject.

In [ ]:
def outer_splits(y, seed=OUTER_SEED, n_splits=N_OUTER):
    """StratifiedKFold(10, shuffle, seed) on y: list of (train_idx, test_idx)."""
    return list(StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(np.zeros((len(y), 1)), y))

def split_es(y, idx, seed, frac=ES_FRACTION):
    """Training subjects `idx` -> (fit, early-stopping), stratified."""
    idx = np.asarray(idx)
    fit, es = train_test_split(idx, test_size=frac, stratify=y[idx], random_state=seed)
    return np.sort(fit), np.sort(es)

def inner_splits(y, idx, seed=INNER_SEED, n_splits=N_INNER):
    """Inner CV restricted to the outer-train subjects `idx` (absolute indices)."""
    idx = np.asarray(idx)
    sp = StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(np.zeros((len(idx), 1)), y[idx])
    return [(idx[a], idx[b]) for a, b in sp]

def fit_scaler(X, idx):
    """RobustScaler fitted on the subjects `idx` only (structural + functional columns)."""
    # ---- EXERCISE ---- Easy
    # Fit a RobustScaler on the rows `idx` of X only (the training subjects) and return the fitted scaler
    return ...
    # ------------------

def scaled(sc, X, idx):
    """(X_struct, X_funct) of the subjects `idx`, scaled with `sc`, float32."""
    Z = sc.transform(X[np.asarray(idx)]).astype(np.float32)
    return Z[:, :N_STRUCT], Z[:, N_STRUCT:]

def shap_background_idx(y, train_idx, fold, n=100):
    """n training subjects, stratified, used as SHAP background."""
    bg, _ = train_test_split(np.asarray(train_idx), train_size=n, stratify=y[train_idx], random_state=OUTER_SEED + fold)
    return np.sort(bg)

In [ ]:
# ---- EXERCISE ---- Easy
# Build the outer folds by applying `outer_splits` to the labels stored in the data dict D, then print the number of
# training and test subjects of every fold as a list of (n_train, n_test)
OUTER = ...
print('fold sizes (train, test):\n', ...)
# ------------------


**Leakage check on fold 0.** We assert that the test subjects appear in no training set, and show that the scaler fitted on the training subjects has a different median than a scaler fitted on all the subjects (the small difference is exactly the information that would leak otherwise).

In [ ]:
tr0, te0 = OUTER[0]
fit0, es0 = split_es(D['y'], tr0, seed=OUTER_SEED + 0)
# ---- EXERCISE ---- Medium
# Check that there is no leakage:
# - the test subjects te0 appear neither in tr0, nor in fit0, nor in es0
# - fit0 and es0 together are exactly tr0 (no subject lost or duplicated)
# Hint: assert, np.intersect1d, np.array_equal
...
# ------------------

sc_train = fit_scaler(D['X'], tr0)
sc_all = fit_scaler(D['X'], np.arange(len(D['y'])))
print(f'fold 0: train {len(tr0)} = fit {len(fit0)} + early-stopping {len(es0)} | test {len(te0)} | overlap test/train: 0')
print('max |median(train) - median(all)| over features: %.4f' % np.abs(sc_train.center_ - sc_all.center_).max())

## 3. From a connectivity array to a (sparse) graph

**The kNN rule** (function `knn_keep`). For one subject, take the absolute connectivity $|r_{ij}|$ between all pairs of regions.

1. Every region keeps its $k$ strongest connections (the diagonal is excluded) (the strongest in our case);
2. The graph is made undirected by **union**: connection $(i, j)$ is kept if $j$ is among the $k$ strongest of $i$ **or** $i$ is among the $k$ strongest of $j$.

So every region has *at least* $k$ neighbours, and a few **hub regions** can have many more. The rule uses only that subject's own values (no label, no other subject), so the graph is a deterministic function of the subject and **cannot leak** information across subjects or folds. The mask is a non-differentiable selection: gradients flow through the kept values only.

**Building the batch** (function `dense_graph_batch`). From the flat vector of a subject we can build:

- The **node features** $x \in \mathbb{R}^{103 \times 103}$: node $i$ is described by its row of the connectivity matrix (its connectivity profile);
- The **edges**: both directions of every kept connection, with the connection value as `edge_attr`.

In the **complete** graph all 5253 connections are edges (10 506 directed edges). In the **kNN strict** graph only the kept connections are edges *and* the removed ones are set to 0 in $x$ (after scaling, 0 is the training median), so a removed connection is invisible both to message passing and to the node profiles.

In [ ]:
def triu_edge_index(R=N_REGIONS):
    """Directed edge_index (2, 2E) of the complete graph (both directions of every pair) and the functional-feature id
    of every edge."""
    iu, ju = feature_pairs(R)
    feat_id = np.arange(iu.shape[0])
    src, dst = np.concatenate([iu, ju]), np.concatenate([ju, iu])
    return (torch.tensor(np.vstack([src, dst]), dtype=torch.long),
            torch.tensor(np.concatenate([feat_id, feat_id]), dtype=torch.long))

def vec_to_matrix(vec, R=N_REGIONS):
    """Symmetric R x R connectivity matrix (zero diagonal) from a 1-D feature vector."""
    M = np.zeros((R, R), dtype=np.float32)
    M[IU, JU] = vec
    return M + M.T

# ---- EXERCISE ---- Medium
# Write the entire knn_keep function given its inputs (all torch tensors, batched over B subjects):
# - abs_r [B, F] is |r|, the absolute Pearson correlation between the time series of each pair of regions
# - k is the number of retained neighbours per region (here k = 1)
# - iu, ju are the row / column indices of the upper triangle: connection f joins regions iu[f] and ju[f]
# - R is the number of regions (103 here, it changes with the atlas)
# It returns keep [B, F] bool: True for the connections kept by the kNN rule (see the markdown above).
def knn_keep(abs_r, k, iu, ju, R):
    """torch. abs_r [B, F]: |r| of every connection of B subjects -> keep [B, F] bool.
    Every region keeps its k strongest connections; symmetrised by union (every region: >= k neighbours)."""
    # 1. Create a zero matrix with shape [B, R, R] (same dtype / device as abs_r)
    A = ...
    # 2. Fill the upper triangular positions with the correlations
    A[:, iu, ju] = ...
    # 3. Symmetrize the matrix
    A = A + ...
    # 4. Exclude self-connections: a region is never its own neighbour
    d = ...
    A[:, d, d] = -1.0
    # 5. Select the k strongest neighbours of each region (indices along the last dimension). Hint: Tensor.topk
    top = ...
    # 6. Create a boolean [B, R, R] mask, True at the selected neighbours. Hint: Tensor.scatter_
    M = ...
    # 7. Symmetrize the mask using the union (i -> j or j -> i)
    M = ...
    # 8. Return only the upper triangular connections
    return M[:, iu, ju]
# ------------------

def knn_feature_mask(X_raw, k, chunk=256):
    """numpy. X_raw [n, F] (harmonised, UNSCALED connectivity) -> [n, F] bool kNN masks, one per subject."""
    X_raw = np.asarray(X_raw, dtype=np.float32)
    iu_t, ju_t = torch.as_tensor(IU, dtype=torch.long), torch.as_tensor(JU, dtype=torch.long)
    out = np.zeros(X_raw.shape, dtype=bool)
    for s in range(0, len(X_raw), chunk):
        out[s:s + chunk] = knn_keep(torch.as_tensor(np.abs(X_raw[s:s + chunk])), k, iu_t, ju_t, N_REGIONS).numpy()
    return out

def knn_masks(k):
    """[N, 5253] kNN masks of all the subjects for a given k (cached). Each row uses only its own subject."""
    cache = D.setdefault('_knn', {})
    if k not in cache:
        cache[k] = knn_feature_mask(D['X'][:, N_STRUCT:], int(k))
    return cache[k]

def dense_graph_batch(x_funct, ei, ef, iu, ju, R, keep=None, node_mask=None):
    """Batched subject graphs built on the device from the flat (scaled) functional vectors.
    keep [B, F] bool selects the edges (both directions); node_mask [B, F] multiplies the values used as node
    features (strict graphs: node_mask = keep). Returns x [B*R, R], edge_index, edge_attr [E,1], batch [B*R],
    edge_feat [E], edge_subject [E]."""
    B = x_funct.shape[0]
    node_vec = x_funct if node_mask is None else x_funct * node_mask
    M = x_funct.new_zeros(B, R, R)
    M[:, iu, ju] = node_vec
    M = M + M.transpose(1, 2)
    x = M.reshape(B * R, R)
    E2 = ei.shape[1]
    offs = (torch.arange(B, device=x.device) * R).repeat_interleave(E2)
    edge_index = ei.repeat(1, B) + offs
    edge_attr = torch.cat([x_funct, x_funct], dim=1).reshape(-1, 1)
    edge_feat = ef.repeat(B)
    edge_subject = torch.arange(B, device=x.device).repeat_interleave(E2)
    if keep is not None:
        sel = torch.cat([keep, keep], dim=1).reshape(-1)
        edge_index, edge_attr, edge_feat, edge_subject = (edge_index[:, sel], edge_attr[sel], edge_feat[sel],
                                                          edge_subject[sel])
    batch = torch.arange(B, device=x.device).repeat_interleave(R)
    return x, edge_index, edge_attr, batch, edge_feat, edge_subject

EDGE_INDEX, EDGE_FEAT = triu_edge_index()  # complete graph
print('Complete graph: undirected edges', N_FUNC, '| directed edges', EDGE_INDEX.shape[1])

A first look at the rule on five subjects. With `k = 1` every region keeps one connection, so the number of undirected edges is between 52 and 103 (two regions that choose each other share one edge); with larger `k` the graph quickly becomes denser.

In [ ]:
for k in (1, 3, 10):
    m = knn_feature_mask(D['X'][:5, N_STRUCT:], k)
    print(f'k = {k}: undirected edges of 5 subjects = {m.sum(1)}  ({100 * m.mean():.1f} % of the 5253 connections)')
# Self-check of your knn_keep: with k = 1 every subject has between 52 and 103 undirected edges
m1 = knn_feature_mask(D['X'][:50, N_STRUCT:], 1).sum(1)
assert ((m1 >= N_REGIONS // 2 + 1) & (m1 <= N_REGIONS)).all(), m1
print('knn_keep: OK')


## 4. The JF++ model

**Functional branch: `FunctionalGAT`.** Two `GATv2Conv` layers (8 heads in the first one) act on the subject graph. The node feature of region $i$ is its connectivity profile, the edge feature is the connection value. Layer 1 computes, for every node, an **attention distribution over its neighbours** (softmax over the incoming edges): in the complete graph a node distributes attention over 102 neighbours, in a `k = 1` graph often over one or two. A mean pooling over the 103 nodes gives the graph embedding.

**Structural branch: `MorphologicalTransformer`.** A feature-tokenizer transformer: every one of the 221 structural features becomes a token (`value * weight + bias`), a `[CLS]` token is prepended, and the `[CLS]` output is the embedding. The `[CLS] -> feature` attention of the first block is a per-feature importance.

**Fusion.** `JointFusionModel` concatenates the two embeddings and applies `Linear -> BatchNorm -> ELU -> Dropout -> Linear`. The two single-modality models, `MorphOnlyModel` and `FuncOnlyModel`, are the same encoders followed by the same head.

In [ ]:
class FunctionalGAT(nn.Module):
    def __init__(self, in_dim, hidden=64, out_dim=16, heads=4, dropout=0.3):
        super().__init__()
        # ---- EXERCISE ---- Medium
        # Write the two graph attention layers. Look for GATv2Conv in the PyG documentation
        # (https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.GATv2Conv.html)
        # and find the arguments for:
        # - gat1: in_dim -> hidden features per head, `heads` heads (concatenated), attention dropout `dropout`,
        #         1-D edge features (the connection value), NO self-loops
        # - gat2: (hidden * heads) -> out_dim, a single head, same dropout / edge features / no self-loops
        self.gat1 = ...
        self.gat2 = ...
        # ------------------
        # ---- EXERCISE ---- Easy
        # BatchNorm1d after the first layer: its input is the concatenation of the `heads` outputs of size `hidden`
        self.norm1 = nn.BatchNorm1d(...)
        # ------------------

    def forward(self, x, edge_index, edge_attr, batch, return_attention=False):
        """g [B, out_dim], or (g, alpha) with alpha [E_total] = attention of the FIRST GAT layer (heads averaged):
        edge i->j refers to connection (i, j)."""
        x, (_, alpha) = self.gat1(x, edge_index, edge_attr=edge_attr, return_attention_weights=True)
        # ---- EXERCISE ---- Easy
        # Activation (ELU) after the norm, and the graph readout: mean over the nodes of every subject
        # (Hint: F.elu, global_mean_pool(x, batch))
        x = ...(self.norm1(x))
        x = F.elu(self.gat2(x, edge_index, edge_attr=edge_attr))
        g = ...
        # ------------------
        if return_attention:
            return g, alpha.mean(dim=1)
        return g

class AttnBlock(nn.Module):
    """One transformer block that also returns its attention map."""
    def __init__(self, d, heads, dropout):
        super().__init__()
        # ---- EXERCISE ---- Medium
        # Multi-head self-attention on tokens of size d with `heads` heads and dropout `dropout`; the inputs are
        # shaped [B, tokens, d] (batch first). Hint: nn.MultiheadAttention
        self.attn = ...
        # ------------------
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d, d))

    def forward(self, x):
        h = self.ln1(x)
        a, attn_w = self.attn(h, h, h, need_weights=True, average_attn_weights=True)
        x = x + a
        x = x + self.ff(self.ln2(x))
        return x, attn_w

class MorphologicalTransformer(nn.Module):
    """Feature-tokenizer transformer with a [CLS] token; the CLS->feature attention of the FIRST block gives
    per-feature importance."""
    def __init__(self, n_features, d=32, heads=4, depth=2, out_dim=16, dropout=0.3):
        super().__init__()
        self.n_features = n_features
        self.weight = nn.Parameter(torch.randn(n_features, d) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d))
        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        self.blocks = nn.ModuleList([AttnBlock(d, heads, dropout) for _ in range(depth)])
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, out_dim), nn.ELU())

    def forward(self, x, return_attention=False):
        B = x.shape[0]
        # ---- EXERCISE ---- Easy
        # Complete the model:
        # - feature tokenizer: every feature value becomes a token `value * weight + bias` -> [B, n_features, d]
        #   (Hint: x.unsqueeze(-1) broadcasts against self.weight [n_features, d])
        # - prepend the [CLS] token to the feature tokens -> [B, 1 + n_features, d] (Hint: torch.cat on dim=1)
        # - the embedding is the head applied to the output of the [CLS] token (position 0)
        tokens = ...
        cls = self.cls.expand(B, -1, -1)
        h = ...
        h, attn_1 = self.blocks[0](h)        # block 1: token i is still feature i alone
        for blk in self.blocks[1:]:
            h, _ = blk(h)
        emb = self.head(...)
        # ------------------
        if return_attention:
            return emb, attn_1[:, 0, 1:]
        return emb

class JointFusionModel(nn.Module):
    """GAT (functional) + FT-Transformer (structural) + fusion head; every knob is a constructor kwarg."""
    def __init__(self, n_regions, n_morph, gat_hidden=64, gat_heads=4, gat_out=16, gat_dropout=0.3,
                 morph_d=32, morph_heads=4, morph_depth=2, morph_out=16, morph_dropout=0.3,
                 fusion_hidden=16, fusion_dropout=0.3):
        super().__init__()
        # ---- EXERCISE ---- Medium
        # Define the encoder of each branch from the constructor kwargs: `self.gat` (FunctionalGAT on the
        # n_regions-dimensional node features) and `self.morph` (MorphologicalTransformer on the n_morph features)
        self.gat = ...
        self.morph = ...
        # ------------------
        self.head = nn.Sequential(nn.Linear(gat_out + morph_out, fusion_hidden), nn.BatchNorm1d(fusion_hidden),
                                  nn.ELU(), nn.Dropout(fusion_dropout), nn.Linear(fusion_hidden, 1))

# Morphological only model for testing a single branch
class MorphOnlyModel(nn.Module):
    """Standalone structural classifier around `MorphologicalTransformer`."""

    def __init__(self, n_morph, morph_d=32, morph_heads=4, morph_depth=2, morph_out=16, morph_dropout=0.3,
                 fusion_hidden=16, fusion_dropout=0.3):
        super().__init__()
        # ---- EXERCISE ---- Medium
        # Define the structural encoder `self.morph` (MorphologicalTransformer) from the constructor kwargs
        self.morph = ...
        # ------------------
        self.head = nn.Sequential(nn.Linear(morph_out, fusion_hidden), nn.BatchNorm1d(fusion_hidden),
                                  nn.ELU(), nn.Dropout(fusion_dropout), nn.Linear(fusion_hidden, 1))

    def forward(self, x, return_attention=False):
        if return_attention:
            m, att = self.morph(x, return_attention=True)
        else:
            m = self.morph(x)
        logit = self.head(m).squeeze(-1)
        return (logit, att) if return_attention else logit

# Functional only model for testing a single branch
class FuncOnlyModel(nn.Module):
    """Standalone functional classifier around `FunctionalGAT`."""

    def __init__(self, n_regions, gat_hidden=64, gat_heads=4, gat_out=16, gat_dropout=0.3,
                 fusion_hidden=16, fusion_dropout=0.3):
        super().__init__()
        # ---- EXERCISE ---- Medium
        # Define the functional encoder `self.gat` (FunctionalGAT) from the constructor kwargs
        self.gat = ...
        # ------------------
        self.head = nn.Sequential(nn.Linear(gat_out, fusion_hidden), nn.BatchNorm1d(fusion_hidden),
                                  nn.ELU(), nn.Dropout(fusion_dropout), nn.Linear(fusion_hidden, 1))

### 4.1 Tensor wrappers

The training and evaluation code works on flat arrays `(x_struct [B, 221], x_funct [B, 5253])`. The wrappers hold the graph rule and turn the flat functional vectors into the batched graph **on the device**, then call the model:

- `GraphTensorWrapper` -> joint model; `FuncOnlyTensor` -> functional-only; `MorphOnlyTensor` -> structural-only (no graph).
- `knn_k=None` -> complete graph; `knn_k=k, strict=True` -> kNN strict graph.
- When the graph is sparse, the kNN mask can be **precomputed** (`keep`, used in training to avoid ranking the connections at every step) or rebuilt from `x_funct` by undoing the scaler: `|x_funct * scale + center|` are the original harmonised values. `center` and `scale` are stored as buffers, so a checkpoint carries its own scaler.

In [ ]:
class _GraphBase(nn.Module):
    """Graph rule shared by the wrappers: complete graph (knn_k=None) or per-subject kNN graph rebuilt from
    |x_funct * scale + center| (the fold scaler undone -> harmonised, unscaled values)."""

    def __init__(self, R, edge_index, iu, ju, knn_k=None, strict=False, center=None, scale=None):
        super().__init__()
        self.R, self.knn_k, self.strict = R, (None if knn_k is None else int(knn_k)), bool(strict)
        self.register_buffer('ei', edge_index.clone())
        self.register_buffer('ef', torch.cat([torch.arange(len(iu)), torch.arange(len(iu))]).long())
        self.register_buffer('iu', torch.as_tensor(np.asarray(iu), dtype=torch.long))
        self.register_buffer('ju', torch.as_tensor(np.asarray(ju), dtype=torch.long))
        if knn_k is not None:
            nf = len(np.asarray(iu))
            self.register_buffer('center', torch.zeros(nf) if center is None else
                                 torch.as_tensor(np.asarray(center), dtype=torch.float))
            self.register_buffer('scale', torch.ones(nf) if scale is None else
                                 torch.as_tensor(np.asarray(scale), dtype=torch.float))

    def keep_from(self, x_funct):
        if self.knn_k is None:
            return None
        with torch.no_grad():
            return knn_keep((x_funct * self.scale + self.center).abs(), self.knn_k, self.iu, self.ju, self.R)

    def graph(self, x_funct, keep=None):
        """keep: precomputed [B, F] masks (else rebuilt from x_funct)."""
        if keep is None:
            keep = self.keep_from(x_funct)
        node_mask = keep if (self.strict and keep is not None) else None
        return dense_graph_batch(x_funct, self.ei, self.ef, self.iu, self.ju, self.R, keep=keep, node_mask=node_mask)


class GraphTensorWrapper(_GraphBase):
    """f(x_struct [B,221], x_funct [B,F]) -> logit [B,1] for a JointFusionModel."""
    def __init__(self, model, R, edge_index, iu, ju, knn_k=None, strict=False, center=None, scale=None):
        super().__init__(R, edge_index, iu, ju, knn_k, strict, center, scale)
        self.model = model
    def forward(self, x_struct, x_funct, keep=None, return_attention=False):
        x, ei, ea, b, ef, es = self.graph(x_funct, keep)
        if not return_attention:
            g = self.model.gat(x, ei, ea, b)
            m = self.model.morph(x_struct)
            return self.model.head(torch.cat([g, m], dim=1))
        g, alpha = self.model.gat(x, ei, ea, b, return_attention=True)
        m, s_att = self.model.morph(x_struct, return_attention=True)
        logit = self.model.head(torch.cat([g, m], dim=1))
        return logit, {'alpha': alpha, 'edge_feat': ef, 'edge_subject': es, 'dst': ei[1], 'morph': s_att,
                       'n_nodes': x.shape[0]}

class FuncOnlyTensor(_GraphBase):
    """FuncOnlyModel (standalone GAT) as f(x_struct, x_funct) -> logit [B,1], same graph rule."""
    def __init__(self, model, R, edge_index, iu, ju, knn_k=None, strict=False, center=None, scale=None):
        super().__init__(R, edge_index, iu, ju, knn_k, strict, center, scale)
        self.model = model

    def forward(self, x_struct, x_funct, keep=None, return_attention=False):
        x, ei, ea, b, ef, es = self.graph(x_funct, keep)
        if not return_attention:
            return self.model.head(self.model.gat(x, ei, ea, b))
        g, alpha = self.model.gat(x, ei, ea, b, return_attention=True)
        return self.model.head(g), {'alpha': alpha, 'edge_feat': ef, 'edge_subject': es, 'dst': ei[1],
                                    'n_nodes': x.shape[0]}

class MorphOnlyTensor(nn.Module):
    """MorphOnlyModel (standalone FT-Transformer) as f(x_struct, x_funct) -> logit [B,1]."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x_struct, x_funct=None, keep=None, return_attention=False):
        if return_attention:
            logit, att = self.model(x_struct, return_attention=True)
            return logit.unsqueeze(-1), {'morph': att}
        return self.model(x_struct).unsqueeze(-1)

def make_net(kind, mk, knn_k, strict, sc, seed=None):
    """Fresh network of `kind` in {'joint', 'struct', 'funct'} from the flat model kwargs `mk`. `sc` is the fold
    scaler (needed to rebuild the kNN masks from the scaled values)."""
    if seed is not None:
        torch.manual_seed(seed)
    center, scale = sc.center_[N_STRUCT:], sc.scale_[N_STRUCT:]
    if kind == 'joint':
        return GraphTensorWrapper(JointFusionModel(N_REGIONS, N_STRUCT, **mk), N_REGIONS, EDGE_INDEX, IU, JU,
                                  knn_k, strict, center, scale)
    if kind == 'struct':
        return MorphOnlyTensor(MorphOnlyModel(N_STRUCT, **{a: b for a, b in mk.items()
                                                           if a.startswith(('morph_', 'fusion_'))}))
    return FuncOnlyTensor(FuncOnlyModel(N_REGIONS, **{a: b for a, b in mk.items() if a.startswith(('gat_', 'fusion_'))}),
                          N_REGIONS, EDGE_INDEX, IU, JU, knn_k, strict, center, scale)

A sanity check with default hyperparameters. **Sparsification does not change the number of parameters**: complete and kNN-strict graphs use exactly the same network, only the edges (and the zeroed node features) differ.

In [ ]:
DEFAULT_MK = dict(gat_hidden=64, gat_out=16, gat_dropout=0.3, morph_d=32, morph_out=16, morph_dropout=0.3,
                  fusion_hidden=16, fusion_dropout=0.3, **FIXED_ARCH)

def n_params(m):
    return sum(p.numel() for p in m.parameters())

for kind in ('struct', 'funct', 'joint'):
    row = {g: n_params(make_net(kind, DEFAULT_MK, c['knn_k'], c['strict'], sc_train)) for g, c in GRAPH_CFG.items()}
    print(f'({kind}) trainable parameters:', row)

## 5. The training loop

One loop trains every model (structural, functional, joint): **Optimizer: SGD with momentum** (`torch.optim.SGD(lr, momentum)`), no weight decay. **Loss**: weighted BCE with logits (`pos_weight = n_TD / n_ASD` of the fit subjects) **plus an L1 penalty** on the weight matrices (`l1_penalty`). The L1 term is added to the optimised loss but not to the monitored validation loss. **Scheduler / early stopping**: `ReduceLROnPlateau(factor=0.1, patience=5)` and early stopping (patience 15) on the validation loss of the early-stopping subjects; the best weights are restored. A batch of size 1 is never used (BatchNorm). `Split` keeps the scaled arrays (and the precomputed kNN masks) of a set of subjects on the device.

In [ ]:
# This decorator create a Training Configuration class
# that takes as input lr, batch_size, momementum, ...
@dataclass
class TrainCfg:
    lr: float
    batch_size: int
    momentum: float = 0.9
    l1_lambda: float = 0.0
    optimizer: str = 'sgd'
    epochs: int = MAX_EPOCHS
    patience: int = PATIENCE
    seed: int = 0

class Split:
    """Scaled arrays of a set of subjects on the device (+ per-subject kNN masks for sparse graphs)."""
    def __init__(self, xs, xf, y, keep=None, device=DEVICE):
        t = lambda a: torch.as_tensor(np.asarray(a, np.float32), device=device)
        self.xs, self.xf, self.y = t(xs), t(xf), t(y)
        self.keep = None if keep is None else torch.as_tensor(np.asarray(keep, bool), device=device)

    def __len__(self):
        return len(self.y)

    def batch(self, idx):
        kw = {} if self.keep is None else {'keep': self.keep[idx]}
        return self.xs[idx], self.xf[idx], self.y[idx], kw

def make_split(sc, idx, knn_k):
    """Split of the subjects `idx`, scaled with `sc` (fitted on training subjects only)."""
    xs, xf = scaled(sc, D['X'], idx)
    keep = knn_masks(knn_k)[np.asarray(idx)] if knn_k is not None else None
    return Split(xs, xf, D['y'][np.asarray(idx)], keep)

def l1_penalty(model, lam=0.01):
    """L1 on every weight matrix except the final output layer (also the GATv2 projections, the attention
    in-projections and the feature tokenizer, which are not nn.Linear). Biases, norms, [CLS] and the GATv2
    attention vectors are not penalised."""
    linears = [m for m in model.modules() if isinstance(m, nn.Linear)]
    out_w = linears[-1].weight if linears else None
    reg = 0.0
    for m in model.modules():
        for name, w in m.named_parameters(recurse=False):
            if name.endswith('weight') and w.dim() >= 2 and w is not out_w:
                reg = reg + w.abs().sum()
    return lam * reg

def make_optimizer(model, cfg):
    """SGD + momentum, no weight decay (the L1 penalty is added to the loss)."""
    # ---- EXERCISE ---- Easy
    # Define the optimizer using torch: SGD on the model parameters with learning rate cfg.lr and momentum
    # cfg.momentum, no weight decay. Hint: torch.optim.SGD
    return ...
    # ------------------

@torch.no_grad()
def predict_logits(model, split, batch_size=EVAL_BS):
    model.eval()
    out = []
    for i in range(0, len(split), batch_size):
        idx = torch.arange(i, min(i + batch_size, len(split)), device=split.y.device)
        xs, xf, _, kw = split.batch(idx)
        out.append(model(xs, xf, **kw).reshape(-1))
    return torch.cat(out)

def predict_proba(model, split, batch_size=EVAL_BS):
    # ---- EXERCISE ---- Medium
    # Write the predict probability function: the probability of ASD (numpy array [n]) of every subject of `split`,
    # computed from the logits of `predict_logits`. A NaN probability (diverged model) must become 0.5
    ...
    # ------------------

def fit(make_model, train, val, cfg, device=DEVICE):
    """The single training loop. Weighted BCE + L1, SGD + momentum, ReduceLROnPlateau and early stopping on the
    validation loss of `val` (early-stopping subjects drawn from the training data, never the test fold), best
    weights restored. Returns (model, history, info)."""
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    model = make_model().to(device)
    opt = make_optimizer(model, cfg)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)
    n_pos = float(train.y.sum()); n_neg = len(train) - n_pos
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(n_neg / n_pos, device=device))
    gen = torch.Generator(device='cpu').manual_seed(cfg.seed)
    hist = {k: [] for k in ('loss', 'val_loss', 'val_auc', 'lr')}
    best, best_state, best_epoch, wait = np.inf, None, -1, 0
    t0 = time.time()
    for epoch in range(cfg.epochs):
        model.train()
        perm = torch.randperm(len(train), generator=gen).to(device)
        n_b = len(perm) // cfg.batch_size + (len(perm) % cfg.batch_size > 1)   # never a batch of 1 (BatchNorm)
        tot, n_seen = 0.0, 0
        for b in range(n_b):
            idx = perm[b * cfg.batch_size:(b + 1) * cfg.batch_size]
            xs, xf, yb, kw = train.batch(idx)
            opt.zero_grad(set_to_none=True)
            loss = lossf(model(xs, xf, **kw).reshape(-1), yb)
            (loss + l1_penalty(model, cfg.l1_lambda) if cfg.l1_lambda > 0 else loss).backward()
            opt.step()
            tot += loss.item() * len(idx); n_seen += len(idx)
        logits = predict_logits(model, val)
        vl = lossf(logits, val.y).item()
        if not (np.isfinite(vl) and np.isfinite(tot)):          # diverged: stop, keep the best weights so far
            hist['diverged'] = epoch
            break
        yv = val.y.cpu().numpy()
        va = roc_auc_score(yv, torch.sigmoid(logits).cpu().numpy()) if 0 < yv.sum() < len(yv) else np.nan
        hist['loss'].append(tot / max(n_seen, 1)); hist['val_loss'].append(vl); hist['val_auc'].append(va)
        hist['lr'].append(opt.param_groups[0]['lr'])
        sched.step(vl)
        if vl < best - 1e-6:
            best, best_epoch, wait = vl, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if wait >= cfg.patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    n_ep = max(len(hist['loss']), 1)
    dt = time.time() - t0
    return model, hist, {'epochs': n_ep, 'best_epoch': best_epoch, 'train_s': dt, 's_per_epoch': dt / n_ep,
                         'best_val_loss': best, 'diverged': 'diverged' in hist}

def train_net(kind, mk, tk, knn_k, strict, sc, fit_idx, es_idx, epochs, patience, seed):
    """Train a network of `kind` on the fit subjects, early-stop on the ES subjects. `sc` was fitted on the
    training subjects of the fold."""
    knn_k = None if kind == 'struct' else knn_k
    cfg = TrainCfg(**tk, epochs=epochs, patience=patience, seed=seed)
    return fit(lambda: make_net(kind, mk, knn_k, strict, sc), make_split(sc, fit_idx, knn_k),
               make_split(sc, es_idx, knn_k), cfg)

### 5.1 Checkpoints, evaluation and plotting helpers

A checkpoint stores the weights **and** everything needed to rebuild the network: model kwargs, training kwargs, `k`, the fold scaler (`center_`, `scale_`) and the training history. `load_results` loads the 10 checkpoints of a run, checks that the stored scaler equals a scaler fitted on the training subjects of that fold (no leakage), and evaluates each model on the test fold it never saw. `train_results` trains the 10 folds instead and returns the same per-fold dictionaries, so the rest of the notebook does not depend on where the results come from. Here, you can find also the plotting functions

In [ ]:
def ckpt_path(run_dir, kind, fold):
    """joint -> checkpoints/joint_fold{k}_seed0.pt | struct / funct -> checkpoints/{kind}_only_fold{k}.pt"""
    name = f'joint_fold{fold}_seed0.pt' if kind == 'joint' else f'{kind}_only_fold{fold}.pt'
    return os.path.join(run_dir, name)

def save_ckpt(path, net, kind, mk, tk, knn_k, sc, **extra):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({'kind': kind, 'model_kwargs': mk, 'train_kwargs': tk, 'knn_k': knn_k, 'pair_order': 'column-major',
                'scaler_center': sc.center_, 'scaler_scale': sc.scale_,
                'state_dict': {a: b.detach().cpu() for a, b in net.state_dict().items()}, **extra}, path)

def load_ckpt(path, strict):
    """-> (network in eval mode, checkpoint dict, scaler). `strict` is the graph rule of the run (node features of
    the removed connections zeroed), it is not stored in the checkpoint."""
    ck = torch.load(path, map_location='cpu', weights_only=False)
    assert ck['pair_order'] == 'column-major'
    sc = RobustScaler(); sc.center_, sc.scale_ = ck['scaler_center'], ck['scaler_scale']
    net = make_net(ck['kind'], ck['model_kwargs'], ck['knn_k'], strict, sc)
    net.load_state_dict(ck['state_dict'])
    return net.to(DEVICE).eval(), ck, sc

def evaluate(net, split, y):
    p = predict_proba(net, split)
    fpr, tpr, _ = roc_curve(y, p)
    return {'p': p, 'auc': roc_auc_score(y, p), 'acc': float(((p > 0.5) == y).mean()),
            'tpr': np.interp(INTERP_FPR, fpr, tpr)}

RESULTS_CACHE = {}

def load_results(graph, kind, run_dir=None):
    """Evaluate the 10 pretrained checkpoints of `kind` ('joint', 'struct', 'funct') of the run of `graph` on their
    test folds. Returns one dict per fold: p, auc, acc, tpr, knn_k, epochs, best_epoch, history, params."""
    run_dir = run_dir or RUN_DIRS[graph]
    key = (run_dir, kind)
    if key in RESULTS_CACHE:
        return RESULTS_CACHE[key]
    out = []
    for k, (tr, te) in enumerate(OUTER):
        net, ck, sc = load_ckpt(ckpt_path(run_dir, kind, k), GRAPH_CFG[graph]['strict'])
        ref = fit_scaler(D['X'], tr)
        if not (np.allclose(sc.center_, ref.center_) and np.allclose(sc.scale_, ref.scale_)):
            print(f'  warning, fold {k}: the stored scaler differs from a scaler fitted on the training subjects')
        r = evaluate(net, make_split(sc, te, None if kind == 'struct' else ck['knn_k']), D['y'][te])
        info = ck.get('info', {})
        r.update(fold=k, knn_k=ck['knn_k'], epochs=info.get('epochs'), best_epoch=info.get('best_epoch'),
                 history=info.get('history', {}), params=ck['model_kwargs'], train_kwargs=ck['train_kwargs'])
        out.append(r)
        del net
    RESULTS_CACHE[key] = out
    return out

def summarize(res, name):
    A = np.array([r['auc'] for r in res]); C = np.array([r['acc'] for r in res])
    print(f'{name}: AUC {A.mean():.3f} (+- {A.std():.3f}) | accuracy {C.mean():.3f} (+- {C.std():.3f})')
    return pd.DataFrame({'fold': [r['fold'] for r in res], 'AUC': A, 'accuracy': C,
                         'knn_k': [r['knn_k'] for r in res], 'epochs': [r['epochs'] for r in res]})

def train_results(graph, kind, params, out_dir):
    """Train the 10 folds of `kind` ('joint', 'struct', 'funct') on the graph `graph` with the best parameters of
    each fold, save the checkpoints in `out_dir` and evaluate each model on its test fold. Same output as
    `load_results`."""
    g = GRAPH_CFG[graph]
    out = []
    for k, (tr, te) in enumerate(OUTER):
        t0 = time.time()
        seed0 = OUTER_SEED + k
        mk, tk = params_to_kwargs(params[k])                       # best Optuna parameters of this fold
        sc = fit_scaler(D['X'], tr)                                # scaler on the training subjects only
        fit_idx, es_idx = split_es(D['y'], tr, seed=seed0)
        net, hist, info = train_net(kind, mk, tk, g['knn_k'], g['strict'], sc, fit_idx, es_idx,
                                    MAX_EPOCHS, PATIENCE, seed0)
        info['history'] = hist
        save_ckpt(ckpt_path(out_dir, kind, k), net, kind, mk, tk, g['knn_k'], sc, info=info, seed=seed0)
        r = evaluate(net, make_split(sc, te, None if kind == 'struct' else g['knn_k']), D['y'][te])
        r.update(fold=k, knn_k=g['knn_k'], epochs=info['epochs'], best_epoch=info['best_epoch'], history=hist,
                 params=mk, train_kwargs=tk)
        out.append(r)
        print(f'fold {k + 1:2d}  AUC {r["auc"]:.3f}  acc {r["acc"]:.3f}  epochs {info["epochs"]:3d}  '
              f'({time.time() - t0:.1f}s)')
        del net
    return out

def get_results(kind):
    """Sections 7-9: train the models (TRAIN = True) or load the pretrained checkpoints of the selected graph."""
    if TRAIN:
        return train_results(GRAPH, kind, PARAMS, TRAIN_WEIGHTS)
    return load_results(GRAPH, kind)

def summarize(res, name):
    A = np.array([r['auc'] for r in res]); C = np.array([r['acc'] for r in res])
    print(f'{name}: AUC {A.mean():.3f} (+- {A.std():.3f}) | accuracy {C.mean():.3f} (+- {C.std():.3f})')
    return pd.DataFrame({'fold': [r['fold'] for r in res], 'AUC': A, 'accuracy': C,
                         'knn_k': [r['knn_k'] for r in res], 'epochs': [r['epochs'] for r in res]})

def plot_history(hist, color, title=''):
    """Training curves of one model: weighted BCE, AUC on the early-stopping subjects and learning rate. The
    vertical line marks the epoch whose weights were restored."""
    ep = np.arange(1, len(hist['loss']) + 1)
    best = int(np.nanargmin(hist['val_loss'])) + 1
    fig, ax = plt.subplots(1, 3, figsize=(FIG_W2, 50 * MM), layout='constrained')
    ax[0].plot(ep, hist['loss'], color=MUTED, label='Training')
    ax[0].plot(ep, hist['val_loss'], color=color, label='Early-stopping set')
    ax[0].set_ylabel('Weighted BCE'); ax[0].legend(loc='upper right')
    ax[1].plot(ep, hist['val_auc'], color=color); ax[1].set_ylabel('AUC (early-stopping set)')
    ax[2].step(ep, hist['lr'], where='post', color=color); ax[2].set_yscale('log'); ax[2].set_ylabel('Learning rate')
    for a, lab in zip(ax, 'abc'):
        a.axvline(best, color=MUTED, lw=0.6, ls=(0, (2, 2)), zorder=0)
        a.set_xlabel('Epoch'); a.xaxis.set_major_locator(MaxNLocator(integer=True)); ygrid(a); panel_label(a, lab)
    if title:
        fig.suptitle(title, x=0.01, ha='left', fontsize=8, fontweight='bold')
    return fig

def plot_histories(res, color, title, name=None):
    """Small multiples of the training curves of the 10 folds (shared y axis); the dot marks the restored epoch."""
    fig, axes = plt.subplots(2, 5, figsize=(FIG_W2, 72 * MM), sharey=True, layout='constrained')
    for a, r in zip(axes.flat, res):
        h = r['history']
        if not h or not h.get('loss'):
            a.set_axis_off(); continue
        ep = np.arange(1, len(h['loss']) + 1)
        b = int(np.nanargmin(h['val_loss']))
        a.plot(ep, h['loss'], color=MUTED, lw=1)
        a.plot(ep, h['val_loss'], color=color, lw=1.2)
        a.scatter(ep[b], h['val_loss'][b], s=16, color=color, edgecolor='white', linewidth=0.8, zorder=3)
        a.set_title(f'Fold {r["fold"] + 1}   AUC {r["auc"]:.2f}', fontsize=6.5)
        a.xaxis.set_major_locator(MaxNLocator(nbins=4, integer=True)); ygrid(a)
    fig.supxlabel('Epoch', fontsize=7); fig.supylabel('Weighted BCE', fontsize=7)
    handles = [Line2D([], [], color=MUTED, lw=1, label='Training'),
               Line2D([], [], color=color, lw=1.2, label='Early-stopping set'),
               Line2D([], [], color=color, marker='o', ls='', ms=4, mec='white', label='Restored epoch')]
    fig.legend(handles=handles, loc='outside upper right', ncol=3)
    fig.suptitle(title, x=0.01, ha='left', fontsize=8, fontweight='bold')
    if name:
        save_fig(fig, name)
    return fig

def plot_mean_roc(ax, tprs, aucs, color, label, z=2):
    """Mean ROC curve over the folds (interpolated on INTERP_FPR) with a +-1 SD band."""
    mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
    sd = np.std(tprs, axis=0)
    ax.fill_between(INTERP_FPR, np.maximum(mean_tpr - sd, 0), np.minimum(mean_tpr + sd, 1), color=color, alpha=.14,
                    lw=0, zorder=z)
    ax.plot(INTERP_FPR, mean_tpr, color=color, lw=1.5, zorder=z + 1,
            label=f'{label}  {np.mean(aucs):.2f} ± {np.std(aucs):.2f}')

def style_roc(ax):
    ax.plot([0, 1], [0, 1], color=MUTED, lw=0.6, ls=(0, (3, 2)), zorder=1)
    ax.text(0.62, 0.55, 'Chance', color=MUTED, fontsize=6, rotation=45, ha='center', va='center',
            rotation_mode='anchor')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.0); ax.set_aspect('equal')
    ax.set_xticks([0, .25, .5, .75, 1]); ax.set_yticks([0, .25, .5, .75, 1])
    ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
    ax.legend(loc='lower right', title='AUC (mean ± SD)', title_fontsize=6.5, alignment='left')

def plot_fold_auc(ax, groups, seed=0):
    """Per-fold test AUC of several models (dots) with mean +- SD (point and bar beside them)."""
    rng_j = np.random.default_rng(seed)
    for i, (lab, a, c) in enumerate(groups):
        a = np.asarray(a)
        ax.scatter(i - 0.08 + rng_j.uniform(-0.09, 0.09, len(a)), a, s=14, color=c, alpha=0.6,
                   edgecolor='white', linewidth=0.5, zorder=3)
        ax.errorbar(i + 0.2, a.mean(), a.std(), fmt='o', color=c, ms=4.5, mec='white', mew=0.8, elinewidth=1.2,
                    capsize=0, zorder=4)
    ax.axhline(0.5, color=MUTED, lw=0.6, ls=(0, (3, 2)), zorder=1)
    ax.set_xticks(range(len(groups))); ax.set_xticklabels([g[0] for g in groups])
    ax.set_xlim(-0.6, len(groups) - 0.4); ax.set_ylabel('Test AUC (per fold)'); ygrid(ax)


## 6. Hyperparameter search with Optuna (nested)

The search is **nested**: it is repeated inside every outer fold, using only the training subjects of that fold, so the test fold never influences the choice of hyperparameters.

* Per trial: sample the hyperparameters, run a **3-fold inner CV** on the outer-train subjects (scaler fitted on the inner-train subjects only), train with a reduced budget (`TRIAL_EPOCH_CAP` epochs, patience 6) and return the **mean inner AUC**, which Optuna maximises.
* Sampler: TPE (`seed=1`); pruner: `MedianPruner(n_startup_trials=3)`; 100 trials.
* **Searched**: GAT hidden width and output size, GAT dropout, transformer token size / output size / dropout, fusion width and dropout, and the optimizer settings `lr` (log, 1e-4 to 1e-1), `momentum` (0.5 to 0.99), `l1_lambda` (log, 1e-5 to 1e-1), `batch_size` (16, 32, 64). **Not searched**: number of heads and depth (`FIXED_ARCH`) and **the kNN `k`, fixed to 1**.

In [ ]:
def sample_params(trial):
    """Search space of JF++ trained with SGD + momentum + L1. The kNN k is NOT sampled (fixed, KNN_K)."""
    p = dict(gat_hidden=trial.suggest_categorical('gat_hidden', [32, 64, 128]),
             gat_out=trial.suggest_categorical('gat_out', [8, 16, 32]),
             gat_dropout=trial.suggest_float('gat_dropout', 0.1, 0.5),
             morph_d=trial.suggest_categorical('morph_d', [16, 32, 64]),
             morph_out=trial.suggest_categorical('morph_out', [8, 16, 32]),
             morph_dropout=trial.suggest_float('morph_dropout', 0.1, 0.5),
             fusion_hidden=trial.suggest_categorical('fusion_hidden', [8, 16, 32]),
             fusion_dropout=trial.suggest_float('fusion_dropout', 0.1, 0.5),
             lr=trial.suggest_float('lr', 1e-4, 1e-1, log=True),
             momentum=trial.suggest_float('momentum', 0.5, 0.99),
             l1_lambda=trial.suggest_float('l1_lambda', *L1_RANGE, log=True),
             batch_size=trial.suggest_categorical('batch_size', BATCH_GRID))
    assert p['morph_d'] % FIXED_ARCH['morph_heads'] == 0
    return p


def params_to_kwargs(p):
    """Best params of one fold -> (model_kwargs, train_kwargs). A stored knn_k (older runs searched it) is ignored."""
    p = {a: b for a, b in p.items() if a != 'knn_k'}
    mk = dict(gat_hidden=p['gat_hidden'], gat_out=p['gat_out'], gat_dropout=p['gat_dropout'], morph_d=p['morph_d'],
              morph_out=p['morph_out'], morph_dropout=p['morph_dropout'], fusion_hidden=p['fusion_hidden'],
              fusion_dropout=p['fusion_dropout'], **FIXED_ARCH)
    tk = dict(optimizer='sgd', lr=p['lr'], momentum=p['momentum'], l1_lambda=p['l1_lambda'],
              batch_size=p['batch_size'])
    return mk, tk


def params_path(run_dir, fold):
    return os.path.join(run_dir, f'kfold_{fold}_joint_fusion_pp_best-params.json')


def search_fold(fold, tr, knn_k, strict, n_trials=N_TRIALS, out_dir=None):
    """Optuna search of the outer fold `fold` on its training subjects `tr` (inner 3-fold CV, mean inner AUC)."""
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize', study_name=f'hpo_fold{fold}',
                                sampler=optuna.samplers.TPESampler(seed=SAMPLER_SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=N_STARTUP, n_warmup_steps=0))
    inner = inner_splits(D['y'], tr)

    def objective(trial):
        mk, tk = params_to_kwargs(sample_params(trial))
        aucs = []
        for i, (itr, iva) in enumerate(inner):
            fit_idx, es_idx = split_es(D['y'], itr, seed=100 + i)
            sc = fit_scaler(D['X'], itr)                      # inner-train only
            net, _, _ = train_net('joint', mk, tk, knn_k, strict, sc, fit_idx, es_idx,
                                  TRIAL_EPOCH_CAP, TRIAL_PATIENCE, seed=i)
            aucs.append(roc_auc_score(D['y'][iva], predict_proba(net, make_split(sc, iva, knn_k))))
            del net
            trial.report(float(np.mean(aucs)), i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(aucs))

    study.optimize(objective, n_trials=n_trials, gc_after_trial=True)
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)
        with open(params_path(out_dir, fold), 'w') as f:
            json.dump(study.best_params, f)
        study.trials_dataframe().to_csv(params_path(out_dir, fold).replace('_best-params.json', '_trials.csv'), index=False)
    return study


def load_params(run_dir):
    """Best Optuna parameters of the 10 folds (list of dicts), or None if the files are missing."""
    paths = [params_path(run_dir, k) for k in range(N_OUTER)]
    if not all(os.path.exists(p) for p in paths):
        return None
    return [json.load(open(p)) for p in paths]

### 6.1 Run the search (optional, `RUN_SEARCH = True`)

> **Run time.** 10 folds × 100 trials × 3 inner trainings. By default the search is skipped and the best parameters stored in `PARAMS_DIR/<run>` are used. With `RUN_SEARCH = True` the best parameters of each fold are written to `TRAIN_DIR/params/<run>` and used by the training cells.

In [ ]:
if RUN_SEARCH:
    for k, (tr, te) in enumerate(OUTER):
        t0 = time.time()
        study = search_fold(k, tr, G['knn_k'], G['strict'], out_dir=TRAIN_PARAMS)
        print(f'fold {k}: best inner AUC {study.best_value:.4f} | {len(study.trials)} trials | {(time.time() - t0) / 60:.1f} min')
        print('   ', study.best_params)
else:
    print('RUN_SEARCH = False: using the best parameters stored in', PARAM_DIRS[GRAPH])

### 6.2 Load the best parameters of every fold

The table shows the hyperparameters selected independently in each outer fold. The spread across folds tells how stable the choice is (continuous parameters such as `lr` or `l1_lambda` usually vary a lot, which is why each fold keeps its own).

In [ ]:
PARAMS_SRC = TRAIN_PARAMS if RUN_SEARCH else PARAM_DIRS[GRAPH]
PARAMS = load_params(PARAMS_SRC)
if PARAMS is None:
    if TRAIN:
        raise FileNotFoundError(f'no best parameters in {PARAMS_SRC}: run the search (RUN_SEARCH = True) or set TRAIN = False')
    print('no params in', PARAMS_SRC, '-> the checkpoints still carry the hyperparameters they were trained with')
else:
    print('best parameters from', PARAMS_SRC)
    display(pd.DataFrame(PARAMS).rename_axis('fold').round(4))

## 7. Structural branch (FT-Transformer only)

The structural-only model uses the transformer encoder of JF++ and the same head, trained with the hyperparameters selected for the fold. It is a **baseline** that shows how much the 221 morphological features carry alone; it does not depend on the graph.

The next cell runs the full training loop of `train_results` (scaler on the training subjects, early-stopping split, SGD training, checkpoint, test evaluation) or, with `TRAIN = False`, loads the pretrained checkpoints.

In [ ]:
# Train the 10 folds (TRAIN = True) or load the pretrained checkpoints (TRAIN = False)
RES_STRUCT = get_results('struct')

In [ ]:
metrics_struct = summarize(RES_STRUCT, 'structural branch')
plot_histories(RES_STRUCT, C_MODEL['struct'], 'Structural branch (FT-Transformer): training curves of the 10 folds',
               name=f'history_struct_{GRAPH}')
metrics_struct

## 8. Functional branch (GATv2 only)

The functional-only model is the GAT on the subject graph followed by the head. It is the part of the model that **depends on the graph**: with `GRAPH = 'knn_strict'` it sees the `k = 1` graph (default), with `'complete'` all connections. Same loop as before, with the functional wrapper.

In [ ]:
# Train the 10 folds (TRAIN = True) or load the pretrained checkpoints (TRAIN = False)
RES_FUNCT = get_results('funct')

In [ ]:
metrics_funct = summarize(RES_FUNCT, f'functional branch ({GRAPH})')
plot_histories(RES_FUNCT, C_MODEL['funct'], f'Functional branch (GATv2, {LABEL_GRAPH[GRAPH]}): training curves of the 10 folds',
               name=f'history_funct_{GRAPH}')
metrics_funct

## 9. Joint Fusion++

The full model: GAT + FT-Transformer + fusion head, **trained end to end** with a single loss, so the gradient reaches both encoders. Same loop, `kind = 'joint'`. The checkpoint of fold $k$ is `joint_fold{k}_seed0.pt`.

In [ ]:
# Train the 10 folds (TRAIN = True) or load the pretrained checkpoints (TRAIN = False)
RES_JOINT = get_results('joint')

In [ ]:
metrics_joint = summarize(RES_JOINT, f'joint fusion++ ({GRAPH})')
if GRAPH == 'knn_strict' and any(r['knn_k'] != KNN_K for r in RES_JOINT):
    print('note: the checkpoints were trained with k =', sorted({r['knn_k'] for r in RES_JOINT}), '(KNN_K =', KNN_K, ')')
plot_histories(RES_JOINT, C_MODEL['joint'], f'Joint Fusion++ ({LABEL_GRAPH[GRAPH]}): training curves of the 10 folds',
               name=f'history_joint_{GRAPH}')
metrics_joint

### 9.1 Mean ROC curves: structural, functional and joint

Each fold gives one ROC curve, interpolated on a common grid of false-positive rates; panel **a** shows the mean curve and ±1 standard deviation across the 10 folds, panel **b** the test AUC of every fold. The metrics are also saved to CSV (suffix `_trained` when the models were trained in this notebook).

In [ ]:
MODELS = (('struct', RES_STRUCT), ('funct', RES_FUNCT), ('joint', RES_JOINT))
fig, ax = plt.subplots(1, 2, figsize=(FIG_W2, 78 * MM), layout='constrained', gridspec_kw={'width_ratios': [1.15, 1]})
for z, (kind, res) in enumerate(MODELS):
    plot_mean_roc(ax[0], [r['tpr'] for r in res], [r['auc'] for r in res], C_MODEL[kind], LABEL_MODEL[kind], z=2 + 2 * z)
style_roc(ax[0])
ax[0].set_title(f'Mean ROC, {LABEL_GRAPH[GRAPH]} graph')
plot_fold_auc(ax[1], [(LABEL_MODEL_SHORT[kind], [r['auc'] for r in res], C_MODEL[kind]) for kind, res in MODELS])
ax[1].set_title('Test AUC of the 10 folds')
panel_label(ax[0], 'a'); panel_label(ax[1], 'b')
save_fig(fig, f'roc_models_{GRAPH}')

os.makedirs(OUT_DIR, exist_ok=True)
for name, df in (('structural', metrics_struct), ('functional', metrics_funct), ('joint', metrics_joint)):
    df.to_csv(os.path.join(OUT_DIR, f'jfpp_{GRAPH}_{name}_metrics_10kf.csv'), index=False)
pd.DataFrame({'fpr': INTERP_FPR, **{f'tpr_fold{r["fold"]}': r['tpr'] for r in RES_JOINT}}).to_csv(
    os.path.join(OUT_DIR, f'jfpp_{GRAPH}_joint_roc_10kf.csv'), index=False)

## 10. The effect of sparsification

We have used the `k = 1` kNN-strict graph so far. What does sparsification actually do? We study it on **three levels**, always comparing with the complete graph:

1. **The graphs**: how many edges, which degrees, how many connected components remain for each `k` (10.1-10.3).
2. **What the network does with them**: the shape of the GAT attention (10.4) and the computational cost (10.6).
3. **What it costs in accuracy**: the 10-fold AUC of the kNN-strict model against the complete-graph model (10.5), and what happens when the graph rule is changed at inference time on already-trained weights (10.7).

Only the two versions asked for in this exercise are compared: **kNN strict** and **complete**. All the statistics of 10.1-10.3 use the raw (unscaled) connectivity of 300 random subjects and do not need any trained weight.

### 10.1 Size and shape of the graphs as a function of `k`

For each `k` we build the kNN graph of every subject and measure the number of undirected edges, the density (share of the 5253 connections kept), the degree of the nodes, the number of connected components and the mean $|r|$ of the kept connections (the kNN rule keeps the strongest ones, so it is higher than the mean of all connections).

A property worth checking for `k = 1`: every region picks exactly one neighbour, so the graph is a set of trees, each with exactly one pair of mutually-nearest regions. Hence **edges + connected components = 103** (up to ties).

In [ ]:
SWEEP_KS_GRAPH = [1, 2, 3, 5, 7, 10, 15, 20]
rng = np.random.default_rng(0)
SUB = np.sort(rng.choice(len(D['y']), 300, replace=False))        # subjects used for the graph statistics
XF_SUB = D['X'][SUB, N_STRUCT:]
INC = np.zeros((N_FUNC, N_REGIONS), np.float32)
INC[np.arange(N_FUNC), IU] = 1; INC[np.arange(N_FUNC), JU] = 1


def graph_stats(masks, xf_raw):
    """masks [n, F] bool -> per-subject undirected edges, density, degrees, connected components, mean |r| kept."""
    deg = masks.astype(np.float32) @ INC                              # [n, R] degree of every region
    comps = np.empty(len(masks), int)
    for s, m in enumerate(masks):
        A = csr_matrix((np.ones(int(m.sum())), (IU[m], JU[m])), shape=(N_REGIONS, N_REGIONS))
        comps[s] = connected_components(A, directed=False)[0]
    return {'edges': masks.sum(1), 'density': masks.mean(1), 'deg': deg, 'comps': comps,
            'abs_r': (np.abs(xf_raw) * masks).sum(1) / masks.sum(1)}


MASKS_SUB = {k: knn_feature_mask(XF_SUB, k) for k in SWEEP_KS_GRAPH}
MASKS_SUB['complete'] = np.ones((len(SUB), N_FUNC), bool)
GSTAT = {k: graph_stats(m, XF_SUB) for k, m in MASKS_SUB.items()}

rows = []
for k, s in GSTAT.items():
    rows.append({'graph': 'complete' if k == 'complete' else f'kNN strict, k={k}',
                 'undirected edges': f'{s["edges"].mean():.0f} +- {s["edges"].std():.0f}',
                 '% of 5253': round(100 * s['density'].mean(), 2),
                 'mean degree': round(s['deg'].mean(), 1), 'min degree': int(s['deg'].min()),
                 'max degree (mean over subjects)': round(s['deg'].max(1).mean(), 1),
                 'connected components': round(s['comps'].mean(), 1),
                 'mean |r| of kept connections': round(s['abs_r'].mean(), 3)})
display(pd.DataFrame(rows).set_index('graph'))

s1 = GSTAT[1]
print('k = 1: edges + components == 103 in', int((s1['edges'] + s1['comps'] == N_REGIONS).sum()), 'of', len(SUB), 'subjects')

In [ ]:
ks = np.array(SWEEP_KS_GRAPH)
c = C_GRAPH['knn_strict']

def line_sd(ax, x, m, s=None, color=c, marker='o', label=None):
    if s is not None:
        ax.fill_between(x, m - s, m + s, color=color, alpha=.15, lw=0)
    ax.plot(x, m, color=color, marker=marker, ms=3.5, mec='white', mew=0.6, label=label)

def ref_line(ax, y, text, va='bottom'):
    ax.axhline(y, color=MUTED, lw=0.6, ls=(0, (3, 2)), zorder=0)
    ax.annotate(text, (ks[-1], y), xytext=(0, 2 if va == 'bottom' else -2), textcoords='offset points',
                color=INK2, fontsize=6, ha='right', va=va)

fig, ax = plt.subplots(1, 4, figsize=(FIG_W2, 52 * MM), layout='constrained')

line_sd(ax[0], ks, np.array([GSTAT[k]['edges'].mean() for k in ks]), np.array([GSTAT[k]['edges'].std() for k in ks]))
ref_line(ax[0], N_FUNC, 'complete (5253)', va='top')
ax[0].set_yscale('log'); ax[0].set_ylabel('Undirected edges per subject'); ax[0].set_title('Graph size')

line_sd(ax[1], ks, np.array([100 * GSTAT[k]['density'].mean() for k in ks]),
        np.array([100 * GSTAT[k]['density'].std() for k in ks]))
ax[1].set_ylabel('Connections kept (% of 5253)'); ax[1].set_title('Density')

line_sd(ax[2], ks, np.array([GSTAT[k]['deg'].mean() for k in ks]), label='Mean')
line_sd(ax[2], ks, np.array([GSTAT[k]['deg'].max(1).mean() for k in ks]), color=C_MODEL['funct'], marker='s',
        label='Max (hub)')
ref_line(ax[2], N_REGIONS - 1, 'complete (102)', va='top')
ax[2].set_yscale('log'); ax[2].set_ylabel('Node degree'); ax[2].set_title('Degree'); ax[2].legend(loc='lower right')

line_sd(ax[3], ks, np.array([GSTAT[k]['comps'].mean() for k in ks]), np.array([GSTAT[k]['comps'].std() for k in ks]))
ref_line(ax[3], 1, 'complete (1)')
ax[3].set_ylabel('Connected components'); ax[3].set_title('Connectivity')

for a, lab in zip(ax, 'abcd'):
    a.set_xlabel('k (neighbours per region)'); a.set_xticks(ks[[0, 2, 3, 5, 6, 7]]); ygrid(a); panel_label(a, lab)
save_fig(fig, 'graph_size_vs_k')

### 10.2 One subject, complete graph versus kNN strict graphs

Top row: the connectivity matrix as the **node features** see it. For the kNN strict graph, the removed connections are zero, so the matrix becomes sparse (a few non-zero entries per row). Bottom row: the same graphs drawn on a circle, one point per region (red = positive correlation, blue = negative, same colour scale as the matrices; width proportional to $|r|$). The region order on the circle is the index order, not an anatomical one.

In [ ]:
def draw_circular(ax, vec, mask, norm, lw_range=(0.15, 1.1), lw_full=0.1):
    """Graph of one subject on a circle (index order). Edges coloured with CMAP_DIV, width proportional to |r|
    (a fixed thin width when there are more than 1000 edges)."""
    th = np.linspace(0, 2 * np.pi, N_REGIONS, endpoint=False) + np.pi / 2
    xy = np.c_[np.cos(th), np.sin(th)]
    sel = np.flatnonzero(mask)
    sel = sel[np.argsort(np.abs(vec[sel]))]                 # strongest connections drawn on top
    segs = np.stack([xy[IU[sel]], xy[JU[sel]]], axis=1)
    w = np.abs(vec[sel])
    full = len(sel) > 1000
    lo, hi = lw_range
    widths = lw_full if full else lo + (hi - lo) * w / (w.max() or 1)
    ax.add_collection(LineCollection(segs, colors=CMAP_DIV(norm(vec[sel])), linewidths=widths,
                                     alpha=(0.2 if full else 0.85), capstyle='round'))
    ax.scatter(xy[:, 0], xy[:, 1], s=3, color=INK2, lw=0, zorder=3)
    ax.set_xlim(-1.08, 1.08); ax.set_ylim(-1.08, 1.08); ax.set_aspect('equal'); ax.axis('off')
    ax.text(0, -1.2, f'{len(sel)} edges', ha='center', va='top', fontsize=6.5, color=INK2)

SHOW_KS = [1, 3, 10]
i_show = 0                                             # index inside SUB
vec = XF_SUB[i_show]
variants = [('Complete', MASKS_SUB['complete'][i_show])] + [(f'kNN strict, k = {k}', MASKS_SUB[k][i_show]) for k in SHOW_KS]
lim = np.abs(vec).max()
norm = mpl.colors.Normalize(-lim, lim)
fig, axes = plt.subplots(2, len(variants), figsize=(FIG_W2, 100 * MM), layout='constrained')
for j, (name, m) in enumerate(variants):
    im = axes[0, j].imshow(vec_to_matrix(vec * m), cmap=CMAP_DIV, norm=norm, interpolation='nearest')
    axes[0, j].set_title(name)
    axes[0, j].set_xticks([0, 50, 100]); axes[0, j].set_yticks([0, 50, 100])
    axes[0, j].set_xlabel('Region')
    if j == 0:
        axes[0, j].set_ylabel('Region')
    for s in axes[0, j].spines.values():
        s.set_visible(True); s.set_linewidth(0.4); s.set_color(AXIS)
    draw_circular(axes[1, j], vec, m, norm)
panel_label(axes[0, 0], 'a', x=-0.3); panel_label(axes[1, 0], 'b', x=-0.3, y=1.0)
cb = fig.colorbar(im, ax=axes, shrink=0.6, aspect=25, pad=0.02)
cb.set_label('Functional connectivity (harmonised)'); cb.outline.set_linewidth(0.4); cb.ax.tick_params(width=0.4)
save_fig(fig, 'graph_example_subject')

### 10.3 Are the sparse graphs the same for every subject?

The kNN graph is computed **per subject** from its own connectivity, so it is not a fixed anatomical skeleton. Two questions:
- **Overlap**: for random pairs of subjects, what is the Jaccard index of their edge sets (1 = identical graphs)?
- **Coverage**: how many of the 5253 connections are an edge in *at least one* subject, and in at least 5 % of the subjects?

A low overlap with high coverage means that sparsification selects *different* strong connections in different people, so the model still sees the whole connectome across the dataset even though each single graph is very small.

In [ ]:
def pairwise_jaccard(masks, n_pairs=3000, seed=0):
    r = np.random.default_rng(seed)
    i, j = r.integers(0, len(masks), n_pairs), r.integers(0, len(masks), n_pairs)
    ok = i != j
    i, j = i[ok], j[ok]
    return (masks[i] & masks[j]).sum(1) / (masks[i] | masks[j]).sum(1)

jac = {k: pairwise_jaccard(MASKS_SUB[k]) for k in SWEEP_KS_GRAPH}
pres = {k: MASKS_SUB[k].mean(0) for k in SWEEP_KS_GRAPH}
ks = np.array(SWEEP_KS_GRAPH)
c = C_GRAPH['knn_strict']

fig, ax = plt.subplots(1, 3, figsize=(FIG_W2, 56 * MM), layout='constrained')
m, s = np.array([jac[k].mean() for k in ks]), np.array([jac[k].std() for k in ks])
ax[0].fill_between(ks, m - s, m + s, color=c, alpha=.15, lw=0)
ax[0].plot(ks, m, color=c, marker='o', ms=3.5, mec='white', mew=0.6)
ax[0].set_xlabel('k (neighbours per region)'); ax[0].set_ylabel('Jaccard index of two subjects')
ax[0].set_title('Overlap between subjects'); ax[0].set_ylim(0, None)

ax[1].plot(ks, [100 * (pres[k] > 0).mean() for k in ks], color=c, marker='o', ms=3.5, mec='white', mew=0.6,
           label='In ≥ 1 subject')
ax[1].plot(ks, [100 * (pres[k] >= 0.05).mean() for k in ks], color=C_MODEL['funct'], marker='s', ms=3.5, mec='white',
           mew=0.6, label='In ≥ 5 % of subjects')
ax[1].set_xlabel('k (neighbours per region)'); ax[1].set_ylabel('Connections (% of 5253)')
ax[1].set_title('Coverage across subjects'); ax[1].set_ylim(0, 102); ax[1].legend(loc='lower right')

ax[2].hist(100 * pres[1][pres[1] > 0], bins=40, color=c, edgecolor='white', linewidth=0.4)
ax[2].set_yscale('log'); ax[2].set_xlabel('Subjects with this edge (%, k = 1)')
ax[2].set_ylabel('Connections'); ax[2].set_title('Selection frequency (k = 1)')
for a, lab in zip(ax, 'abc'):
    ygrid(a); panel_label(a, lab)
for a in ax[:2]:
    a.set_xticks(ks[[0, 2, 3, 5, 6, 7]])
save_fig(fig, 'graph_overlap_coverage')

### 10.4 AUC: kNN strict against complete

This comparison always uses the **provided** hyperparameters and weights of both runs (`../params/<run>` and `../weights/<run>`), whatever `TRAIN` is. We first check that the hyperparameters stored in every checkpoint are the ones in its params folder.

Both models were tuned with the same nested Optuna protocol on the same ten folds, so the AUCs are **paired by fold**. We show the per-fold AUC, the paired difference and a paired Wilcoxon test. With only ten folds the test has little power: the mean difference and its spread across folds are as informative as the p-value.

In [ ]:
CMP = ('complete', 'knn_strict')
JOINT = {g: load_results(g, 'joint', RUN_DIRS[g]) for g in CMP}       # pretrained weights of both runs
PARAMS_CMP = {g: load_params(PARAM_DIRS[g]) for g in CMP}            # their best Optuna parameters

def same_kwargs(stored, ref):
    return all(np.isclose(stored[a], b) if isinstance(b, float) else stored[a] == b for a, b in ref.items())

for g in CMP:
    if PARAMS_CMP[g] is None:
        print(f'{g:11s} no params in {PARAM_DIRS[g]}')
        continue
    ok = [same_kwargs({**r['params'], **r['train_kwargs']}, {**mk, **tk})
          for r, (mk, tk) in zip(JOINT[g], map(params_to_kwargs, PARAMS_CMP[g]))]
    print(f'{g:11s} weights: {RUN_DIRS[g]} | params: {PARAM_DIRS[g]} | hyperparameters match in {sum(ok)}/{len(ok)} folds')

auc_g = {g: np.array([r['auc'] for r in JOINT[g]]) for g in CMP}
acc_g = {g: np.array([r['acc'] for r in JOINT[g]]) for g in CMP}
diff = auc_g['knn_strict'] - auc_g['complete']

for g in CMP:
    print(f'{g:11s} AUC {auc_g[g].mean():.3f} (+- {auc_g[g].std():.3f}) | accuracy {acc_g[g].mean():.3f} (+- {acc_g[g].std():.3f})')
print(f'AUC difference (strict - complete): {diff.mean():+.3f} (+- {diff.std():.3f}); strict better in {(diff > 0).sum()}/{len(diff)} folds')
p_w = stats.wilcoxon(auc_g['knn_strict'], auc_g['complete']).pvalue if np.any(diff != 0) else np.nan
print(f'paired Wilcoxon (two-sided): p = {p_w:.3f}')

fig, ax = plt.subplots(1, 3, figsize=(FIG_W2, 66 * MM), layout='constrained', gridspec_kw={'width_ratios': [0.8, 1, 1.2]})

# a: paired AUC per fold
for k in range(N_OUTER):
    ax[0].plot([0, 1], [auc_g['complete'][k], auc_g['knn_strict'][k]], color=AXIS, lw=0.7, zorder=1)
for x, g in enumerate(CMP):
    ax[0].scatter(np.full(N_OUTER, x), auc_g[g], s=14, color=C_GRAPH[g], edgecolor='white', linewidth=0.5, zorder=3)
    ax[0].errorbar(x + (0.18 if x else -0.18), auc_g[g].mean(), auc_g[g].std(), fmt='o', color=C_GRAPH[g], ms=4.5,
                   mec='white', mew=0.8, elinewidth=1.2, capsize=0, zorder=4)
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['Complete', f'kNN strict\n(k = {KNN_K})']); ax[0].set_xlim(-0.45, 1.45)
ax[0].set_ylabel('Test AUC'); ax[0].set_title('Paired by fold'); ygrid(ax[0])
ax[0].text(0.5, 1.0, 'Wilcoxon p < 0.001' if p_w < 1e-3 else f'Wilcoxon p = {p_w:.3f}', transform=ax[0].transAxes, ha='center', va='top', fontsize=6.5,
           color=INK2)

# b: paired difference per fold
folds = np.arange(1, N_OUTER + 1)
ax[1].bar(folds, diff, width=0.7, color=C_GRAPH['knn_strict'], edgecolor='white', linewidth=0.4, zorder=2)
ax[1].axhline(0, color=INK2, lw=0.6, zorder=3)
ax[1].axhline(diff.mean(), color=INK, lw=0.8, ls=(0, (3, 2)), zorder=3)
ax[1].set_xlim(0.4, N_OUTER + 2.2)
ax[1].annotate(f'mean\n{diff.mean():+.3f}', (N_OUTER + 0.5, diff.mean()), xytext=(2, 0), textcoords='offset points',
               ha='left', va='center', fontsize=6, color=INK)
ax[1].set_xticks(folds); ax[1].set_xlabel('Fold'); ax[1].set_ylabel('ΔAUC (kNN strict − complete)')
ax[1].set_title('Difference per fold'); ygrid(ax[1])

# c: mean ROC
for z, g in enumerate(CMP):
    plot_mean_roc(ax[2], [r['tpr'] for r in JOINT[g]], auc_g[g], C_GRAPH[g], LABEL_GRAPH[g], z=2 + 2 * z)
style_roc(ax[2]); ax[2].set_title('Joint Fusion++, mean ROC')

for a, lab in zip(ax, 'abc'):
    panel_label(a, lab)
save_fig(fig, 'auc_knn_strict_vs_complete')